# PCU-LOCALITY-WIDTH-001 — L7 mutation-width diagnostic

Engineering-only causal diagnostic. Reuses the published L7/K=8 layer-placement result and changes **only the number of selected L7 Cells**.

Primary probes: K=16 and K=32 in parallel on T4 x2. K=64 runs only if neither primary width reaches 80% A direct accuracy. Loss, AdamW, LR=1e-3, 128 steps, batch=8, dataset, seed, routing and evaluation remain unchanged. Formal seeds are never executed.

Requirements: Kaggle Internet ON, **T4 x2**, Secrets `HF_TOKEN` and `GITHUB_TOKEN`.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

BRANCH = 'codex/pcu-composability-kill-001'
REPO = Path('/kaggle/working/mini-cells')
OUT = REPO / 'artifacts/research/pcu-locality-width-001/engineering/26090501-l7-width'
LAYER_BASELINE = REPO / 'artifacts/research/pcu-layer-placement-001/engineering/26090501-layer-only'
FORMAL_SEEDS = (26090511, 26090512, 26090513)
REQUIRED_TRANSFORMERS = '5.16.1'
os.environ.setdefault('HF_HOME', '/kaggle/working/hf-cache')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

def run(cmd, *, env=None, capture=False):
    cmd = [str(x) for x in cmd]
    print('+', ' '.join(cmd))
    result = subprocess.run(cmd, check=True, env=env, text=True, capture_output=capture)
    return result.stdout.strip() if capture else ''

if not REPO.exists():
    run(['git', 'clone', '--branch', BRANCH, 'https://github.com/ArcheLabs/mini-cells.git', REPO])
os.chdir(REPO)
run(['git', 'fetch', 'origin'])
run(['git', 'checkout', BRANCH])
run(['git', 'pull', '--ff-only', 'origin', BRANCH])
run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'])
run([sys.executable, '-m', 'pip', 'install', f'transformers=={REQUIRED_TRANSFORMERS}', 'huggingface_hub>=0.36,<2.0', 'safetensors>=0.4', 'accelerate>=1.0'])

import torch, transformers
assert transformers.__version__ == REQUIRED_TRANSFORMERS
assert torch.cuda.is_available()
assert torch.cuda.device_count() >= 2, f'Need T4 x2; found {torch.cuda.device_count()} CUDA device(s)'
print(json.dumps({
    'commit': run(['git', 'rev-parse', 'HEAD'], capture=True),
    'tree': run(['git', 'rev-parse', 'HEAD^{tree}'], capture=True),
    'gpu0': torch.cuda.get_device_name(0),
    'gpu1': torch.cuda.get_device_name(1),
    'transformers': transformers.__version__,
}, indent=2))


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
github_token = secrets.get_secret('GITHUB_TOKEN')
assert hf_token and github_token
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
os.environ['GITHUB_TOKEN'] = github_token
login(token=hf_token, add_to_git_credential=False)
print('Secrets loaded; token values were not printed.')


In [ ]:
SEED_REGISTRY = REPO / 'research/formal_seed_registry.json'
def formal_states():
    payload = json.loads(SEED_REGISTRY.read_text())
    return {int(row['seed']): row['state'] for row in payload['seeds']}
expected = {seed: 'RESERVED_UNTOUCHED' for seed in FORMAL_SEEDS}
assert formal_states() == expected
layer_decision_path = LAYER_BASELINE / 'DECISION.json'
layer7_path = LAYER_BASELINE / 'LAYER_07.json'
assert layer_decision_path.is_file() and layer7_path.is_file(), 'Published layer-placement baseline is required.'
layer_decision = json.loads(layer_decision_path.read_text())
layer7 = json.loads(layer7_path.read_text())
assert layer_decision['status'] == 'LAYER_PLACEMENT_DID_NOT_RESCUE'
assert layer7['layer'] == 7 and layer7['allocation']['selected_k'] == 8
assert layer7['direct_accuracy'] == 0.046875
print(json.dumps({
    'baseline_status': layer_decision['status'],
    'l7_k8_accuracy': layer7['direct_accuracy'],
    'l7_effective_count': layer7['allocation']['effective_count'],
    'l7_top8_mass': layer7['allocation']['topk_mass']['8'],
    'formal_seed_states': formal_states(),
}, indent=2))


In [ ]:
test_env = os.environ.copy()
test_env['PYTHONPATH'] = str(REPO / 'src')
test_env['PYTEST_DISABLE_PLUGIN_AUTOLOAD'] = '1'
run([sys.executable, '-m', 'pytest', '-q', 'tests/research/05-pcu-kill-001'], env=test_env)
run([sys.executable, '-m', 'compileall', '-q', 'src/minicells/pcu_kill_001', 'scripts/research'])
print('PCU locality-width test/compile gate: PASS')


In [ ]:
existing = sorted(OUT.glob('WIDTH_*.json')) if OUT.exists() else []
print(json.dumps({'resume': bool(existing), 'completed_width_results': [p.name for p in existing]}, indent=2))
run([
    sys.executable, 'scripts/research/run_pcu_locality_width_001.py',
    '--seed', '26090501',
    '--device0', 'cuda:0',
    '--device1', 'cuda:1',
    '--baseline', LAYER_BASELINE,
    '--out', OUT,
])


In [ ]:
required = ['RUN_IDENTITY.json', 'DESIGN.json', 'DECISION.json']
missing = [name for name in required if not (OUT / name).is_file()]
assert not missing, missing
decision = json.loads((OUT / 'DECISION.json').read_text())
design = json.loads((OUT / 'DESIGN.json').read_text())
identity = json.loads((OUT / 'RUN_IDENTITY.json').read_text())
widths = sorted(OUT.glob('WIDTH_*.json'))
expected_count = 3 if decision['fallback_k64_required'] else 2
assert len(widths) == expected_count, [p.name for p in widths]
assert identity['source']['source_dirty'] is False
assert identity['formal_execution_not_started'] is True
assert decision['scientific_evidence'] is False
assert formal_states() == expected
print(json.dumps({
    'status': decision['status'],
    'fallback_k64_required': decision['fallback_k64_required'],
    'comparison': decision['comparison'],
    'best': decision['best'],
    'rescued': decision['rescued'],
    'improved': decision['improved'],
    'completed_width_results': [p.name for p in widths],
    'formal_seed_states': formal_states(),
}, indent=2))


In [ ]:
run([sys.executable, 'scripts/research/publish_pcu_locality_width_001.py', '--branch', BRANCH])
assert formal_states() == expected
print(json.dumps({'published': True, 'formal_seed_states': formal_states()}, indent=2))
